# Final model-family comparison

One fixed CatBoost configuration with native categories, one fixed LightGBM configuration on the existing 44 engineered features, and the saved top-25 XGBoost baseline. Exact reproduction of the original training and validation arrays is required before fitting. No test evaluation or resampling.

Uses the existing `churn-xgboost-weighted-improvement` MLflow experiment. Early stopping uses unweighted average precision. Thresholds maximize validation F1. A single equal-weight ensemble is eligible only if a new model matches baseline AP and has residual correlation below 0.95. No hyperparameter or ensemble-weight search.

Install `experiments/requirements-model-family.txt` in your ML environment before running. The script contains the full reproducible implementation. Baseline time is historical; new model times include fitting and early-stopping evaluation. The existing test split has previously been inspected, although this experiment does not access it.

The completed results are loaded by default. Set `RUN_EXPERIMENT = True` only to explicitly repeat this fixed comparison. Native CatBoost uses seven categorical columns, including a unique-per-training-row signup timestamp. Models and thresholds have not yet been frozen or registered for production.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/processed_datasets.npz").exists())
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    sys.path.insert(0, str(ROOT / "experiments"))
    from model_family_experiment import main
    output_directory = main()
else:
    output_directory = ROOT / "data/model_family_comparison/312c99c0c2f34b16b302bfde4e629c31"
    assert (output_directory / "summary.json").exists()
    print("Loaded completed experiment:", output_directory)


Loaded completed experiment: /home/rami/Desktop/Projects/mlops/data/model_family_comparison/312c99c0c2f34b16b302bfde4e629c31


In [2]:
import pandas as pd
import json
comparison = pd.read_csv(output_directory / "comparison.csv")
columns = ["model", "threshold", "pr_auc", "f1", "precision", "recall", "roc_auc",
           "balanced_accuracy", "accuracy", "fit_seconds", "tn", "fp", "fn", "tp"]
print(comparison[columns].to_string(index=False))
print("\nSummary and limitations:")
print((output_directory / "summary.json").read_text())


           model  threshold   pr_auc       f1  precision   recall  roc_auc  balanced_accuracy  accuracy  fit_seconds    tn    fp   fn   tp
xgboost_baseline   0.511252 0.206725 0.269538   0.190537 0.460455 0.684118           0.622575  0.752546    14.726971 58233 15978 4407 3761
        catboost   0.508413 0.206700 0.270384   0.191047 0.462414 0.684750           0.623454  0.752558   287.916617 58218 15993 4391 3777
        lightgbm   0.508046 0.205935 0.269598   0.189749 0.465475 0.683664           0.623353  0.749924    32.754353 57976 16235 4366 3802

Summary and limitations:
{
  "search_id": "312c99c0c2f34b16b302bfde4e629c31",
  "experiment": "churn-xgboost-weighted-improvement",
  "best_pr_auc": "xgboost_baseline",
  "best_f1": "catboost",
  "best_pr_auc_delta": 0.0,
  "paired_bootstrap_ap_delta_95_interval": [
    -0.0009656191410486105,
    0.000915421341035153
  ],
  "bootstrap_replicates": 200,
  "ensemble_diagnostics": [
    {
      "model": "catboost",
      "residual_correlatio